
# Jellyfish drag force

In this example we illustrate the drag force formula used for Jellyfish.


In [ ]:
import cylag
import numpy as np
import matplotlib.pylab as plt

## Define drag force formulas




In [ ]:
def drag_force_sphere(rhof=1000., Cd=None, dp=0.01, Uf=1., Up=0., nu=1.e-6):
    """ Spherical, Schiller and Nauman (1935) """
    if Cd is None:
        Rep = max(1.e-16, dp*abs(Uf-Up)/nu)
        if Rep <= 1000. and Rep != 0.:
            Cd = (24./(Rep))*(1. + 0.15*(Rep**0.687))
        else:
            Cd = 0.44
    Sp = np.pi*(dp**2.)/4.
    Fd = 0.5*Sp*rhof*Cd*abs(Uf-Up)*(Uf-Up)
    return Fd

def drag_force_jellyfish(rhof=1000., dp=0.5, h=None, L=4., Uf=1., Up=0., nu=1.e-6):
    """ 
    Wang et al. (2026) jellyfish drag model.

    Parameters
    ----------
    dp : float
        Bell diameter [m]
    h : float
        Bell height [m]
        If None, assumes hemispherical bell -> h = dp/2
    L : float
        Tentacle length ratio l/h
  
    """
    h = 0.5*dp if h is None else h
    Re = max(1.e-16, dp*abs(Uf-Up)/nu)

    # Daniel (1983)
    Cf = 1.328/np.sqrt(Re)
    Cd0 = 0.33*(dp/h) + Cf*(3.*h/dp + 3.*np.sqrt(dp/h))

    # Wang et al. (2026)
    a, b = 0.0285, -0.0002
    c1, d1 = 381.4771, 33.1604
    c2, d2 = 125.8822, 1.153211

    c, d = (c1, d1) if Re < 220. else (c2, d2)
    Cd = Cd0*(1. + a*L + b*L**2)*(1. + c/Re + d/np.sqrt(Re))

    Sp = np.pi*(dp**2.)/4.
    Fd = 0.5*Sp*rhof*Cd*abs(Uf-Up)*(Uf-Up)
    return Fd

## Plot drag force




In [ ]:
# compute drag
size = 40
Uf = np.linspace(0., 1., size)
Up = np.zeros(size)
dp = 0.5

Fd_S_0 = np.zeros(size)
Fd_S_1 = np.zeros(size)
Fd_J_0 = np.zeros(size)
Fd_J_1 = np.zeros(size)
Fd_J_2 = np.zeros(size)
for i in range(size):
    Fd_S_0[i] = drag_force_sphere(dp=dp, Cd=None, Uf=Uf[i], Up=Up[i])
    Fd_S_1[i] = drag_force_sphere(dp=dp, Cd=1., Uf=Uf[i], Up=Up[i])
    Fd_J_0[i] = drag_force_jellyfish(dp=dp, Uf=Uf[i], Up=Up[i])

# plot
fig, ax = plt.subplots(1, 1, figsize=(5.5, 4.))
cylag.set_rcparams()
c0, c1, c2 = cylag.get_default_color_palette()
ax.plot(Uf, Fd_S_0, marker='', markersize=4, color=c0[0], lw=1.2, ls='-.', label="Spherical, Schiller and Nauman 1935")
ax.plot(Uf, Fd_S_1, marker='', markersize=4, color=c0[1], lw=1.2, ls='--', label="Spherical ($C_d=1.$)")
ax.plot(Uf, Fd_J_0, marker='s', markersize=3, color=c0[2], lw=1., ls='-', label="Jellyfish ($d=0.5, BDR=0.5, TLR=4$)")
ax.grid()
ax.set_xlabel("$|U_f-U_p|$ (m/s)")
ax.set_ylabel("$F_{d}$ (N)")
ax.legend()
plt.savefig("figs/drag_force_formeCd.png", dpi=300, format="png")
plt.show()